## Notebook for Pre-Training PatchTST

This is the experimental notebook for the PatchTST Pre-Training

First import necessary libraries.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import PatchTSTConfig, PatchTSTModel

/home/dzur/ai-projects/ms-cs-datamining-project/bitcoin-volatility-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data for PatchTST preprocessing.

In [2]:
raw_ds = load_dataset(
    "parquet",
    data_files={
        "train": "../normalized-features/train_df.parquet",
        "validation": "../normalized-features/eval_df.parquet",
    },
)

# Columns
DROP_COLS = ["open_time"]
SYMBOL_COL = "symbol"

Define the preprocessing function.

In [3]:
def preprocess(split_ds, symbol_categories=None):
    """
    Returns:
        features:  np.ndarray (timesteps, channels) -- numeric OHLCV etc.
        symbols:   np.ndarray (timesteps,)          -- raw symbol strings
        categories: list of symbol categories (fitted on train)
    """
    df = split_ds.to_pandas()

    # Drop datetime column
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # Fit symbol categories on the training split only
    if symbol_categories is None:
        symbol_categories = sorted(df[SYMBOL_COL].unique().tolist())

    symbols = df[SYMBOL_COL].values
    features = (
        df.drop(columns=[SYMBOL_COL])
        .select_dtypes(include=[np.number])
        .values.astype(np.float32)
    )

    return features, symbols, symbol_categories

First inspect raw_ds train split to make sure split occured properly.

In [4]:
pd.set_option('display.max_columns', None)

print(raw_ds["train"].to_pandas().head())

                  open_time     open     high      low    close      volume  \
0 2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1 2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2 2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3 2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4 2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   

    symbol  log_ret_1d  log_ret_5d  log_ret_21d  log_ret_63d  log_ret_126d  \
0  BTCUSDT         NaN         NaN          NaN          NaN           NaN   
1  ETHUSDT         NaN         NaN          NaN          NaN           NaN   
2  BTCUSDT         NaN         NaN          NaN          NaN           NaN   
3  ETHUSDT         NaN         NaN          NaN          NaN           NaN   
4  BTCUSDT         NaN         NaN          NaN          NaN           NaN   

   intraday_ret  mom_6_1  hl_log_range  body_to_range  u

In [ ]:
train_features, train_symbols, symbol_categories = preprocess(raw_ds["train"])
val_features, val_symbols, _ = preprocess(
    raw_ds["validation"], symbol_categories=symbol_categories
)

num_channels = train_features.shape[1]